# BERT-Base uncased — DIMER fill-mask and sentence-embedding tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/bert-masked-lm-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/bert-masked-lm-pipeline/blob/main/tutorials/bert_masked_lm_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-google--bert%2Fbert--base--uncased-ffcc4d?style=flat)](https://huggingface.co/google-bert/bert-base-uncased)
[![Upstream](https://img.shields.io/badge/Upstream-google--research%2Fbert-181717?style=flat&logo=github&logoColor=white)](https://github.com/google-research/bert)
[![arXiv](https://img.shields.io/badge/arXiv-1810.04805-b31b1b.svg)](https://arxiv.org/abs/1810.04805)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.0  
**Capability:** masked-language modelling (fill-mask: ranked candidates for one `[MASK]`) and sentence embeddings (768-d, CLS or mean pooled, L2-normalised) using the pinned BERT-Base uncased weights

This notebook is the executable reference path for the repository's two capabilities. It exercises the repository's public pipeline API (`BERTMaskedLMPipeline`) rather than reimplementing model inference. At inference the WordPiece tokenizer lower-cases the text and one forward pass of the 12-layer bidirectional encoder runs; `fill_mask` reads the output-vocabulary logits at the single `[MASK]` position through the masked-LM head and ranks them by a softmax over the 30,522-token vocabulary, while `embed` takes the encoder's last hidden states and pools them to one 768-d vector per text (the `[CLS]` position, or the attention-masked mean) and L2-normalises it. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting — the pinned checkpoint is used as published, with its next-sentence-prediction head and pooler weights left unloaded. What the upstream checkpoint supplies is the encoder, the masked-LM head and the tokenizer; what this repository adds is manifest verification, input validation and ceilings (over-long texts are rejected, not truncated), the two task methods, and fixed output contracts. **The fill-mask `score` is not a calibrated probability** (it is a softmax ranking signal), and **embeddings are representations, not predictions**; the pipeline ships no threshold for either.

**Learning objectives:** bootstrap the repository in a fresh runtime, author a synthetic cloze sentence and a few sentences to embed (or upload your own), surface the pipeline's ceilings, stage and digest-verify the immutable upstream snapshot, run fill-mask and read its ranked candidates correctly, run embedding and read the vector contract correctly (shape, pooling, unit), understand why no metric is reported and what labelled data each capability needs, and export identifiers alongside vectors plus provenance.

**This notebook does not demonstrate:** fine-tuning or classification heads, next-sentence prediction, multi-mask filling, raw-logit access, text generation (BERT is an encoder), cased or non-English text (the checkpoint is uncased English), or contrastively trained sentence similarity (the `qwen3-embedding-pipeline` sibling covers retrieval-grade embeddings). The repository exposes none of these.


## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (also float32; the pipeline loads the checkpoint in float32 on both). The model card's CPU smoke loaded and verified the snapshot in 4.34 s, filled one mask in 0.13 s and embedded two sentences in 0.02 s, so the default runs in seconds on a hosted CPU runtime. The pinned `torch==2.14.0` install and the 440 MB `model.safetensors` are the largest downloads of the run.
- **Knowledge:** basic Python; what a softmax over a vocabulary is and why it is not a calibrated probability; what cosine similarity between unit vectors means.
- **Data:** the default sample is one synthetic cloze sentence and three synthetic sentences authored in code; BYOD is one UTF-8 text file, gated off by default. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded text remains in the notebook runtime; this pipeline does not send it to a third-party inference API.
- **External access:** the Git clone, the pinned wheel installs, and the fetch of the missing snapshot file from the Hugging Face Hub at the immutable revision. No credentials are needed.


## 1. Bootstrap the repository and pinned runtime

When the notebook is opened without a repository checkout, this cell clones the repository. Released notebooks default to `main`; automated candidate validation can set `DIMER_TUTORIAL_REF` to an immutable commit or review branch. The repository is installed as a regular (non-editable) package so it is importable in this same runtime; an editable install would only become importable after a restart. Model-facing dependencies (`torch`, `transformers`, `tokenizers`, `huggingface-hub`, `safetensors`, `numpy`) are pinned exactly by `pyproject.toml`. If installation replaces any package that this runtime has already imported (hosted runtimes commonly pre-import a different NumPy or Pillow), the cell fails with a restart instruction rather than continuing with mixed versions: restart the runtime and rerun from the top. Inference runs in float32 on both CPU and CUDA; no compilation or quantisation is applied. Look for a dictionary reporting the repository revision, Python, `torch`, `transformers` and `numpy` versions, and whether CUDA is available.


In [ ]:
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/kurtvalcorza/bert-masked-lm-pipeline.git'
REPO_NAME = 'bert-masked-lm-pipeline'
REPO_REF = os.environ.get('DIMER_TUTORIAL_REF', 'main')
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '-q', REPO_URL, str(checkout)], check=True)
    if REPO_REF != 'main':
        subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', 'main'], check=True)
        subprocess.run(['git', '-C', str(checkout), 'pull', '--ff-only', '-q', 'origin', 'main'], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    # Every distribution that is already imported in this runtime is captured before installation,
    # whatever its name (PIL -> pillow), so a pinned install that replaces any loaded package is
    # detected. Distribution metadata is compared with metadata afterwards: torch.__version__ carries
    # a local build label (for example 2.14.0+cu130) that the distribution version omits.
    def _installed_version(distribution):
        try:
            return importlib.metadata.version(distribution)
        except importlib.metadata.PackageNotFoundError:
            return None
    _module_dists = importlib.metadata.packages_distributions()
    _loaded_dists = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded_dists}
    # Non-editable install: an editable (.pth) install is not importable until the
    # interpreter restarts, which a fresh hosted runtime cannot do mid-notebook.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(ROOT)], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

REPO_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import platform, numpy, torch, transformers
print({'repository': str(ROOT), 'repository_revision': REPO_SHA, 'requested_ref': REPO_REF, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'numpy': numpy.__version__, 'cuda': torch.cuda.is_available()})

## 2. Author the synthetic sample or optional BYOD

The default sample is **synthetic**, written in this cell: one cloze sentence with a single `[MASK]` (the model card's smoke sentence) together with the token its author expects, and three sentences to embed — two about a pet on a floor covering and one unrelated — each given a stable identifier (`s1`, `s2`, `s3`) so every vector can be mapped back to its text. The expected token is the author's intent, not a labelled dataset: whether it lands in the top-`k` is a smoke/sanity check that the code path works, never an accuracy figure and never benchmark evidence; the three sentences carry **no similarity labels**, so the cosine table they produce is a qualitative check only. `TOP_K` and `POOLING` are Colab form parameters checked against the package in Section 3.

BYOD is optional and disabled by default. Expected BYOD input: one UTF-8 text file whose first non-empty line is a cloze sentence containing exactly one `[MASK]` and whose remaining non-empty lines (at most `MAX_BATCH`) are the sentences to embed, each at most `MAX_TEXT_CHARS` characters and at most `MAX_TEXT_TOKENS` WordPiece tokens (longer texts are rejected by the pipeline, not truncated). Text is lower-cased by the tokenizer, so case carries no information. The upload stays inside this runtime. If you also hold gold tokens or similarity judgements, keep them outside the notebook — Sections 5 and 6 explain what to compute with them.


In [ ]:
import hashlib
import io

USE_BYOD = False  # @param {type:"boolean"}
TOP_K = 5  # @param {type:"integer"}
POOLING = 'mean'  # @param ["cls", "mean"]
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    sample_name = next(iter(uploaded))
    lines = [line.strip() for line in io.StringIO(uploaded[sample_name].decode('utf-8')) if line.strip()]
    if len(lines) < 2:
        raise ValueError(f'{sample_name}: expected a [MASK] sentence line followed by at least one sentence line')
    cloze, sentences = lines[0], lines[1:]
    expected_token = None
    sample_kind = 'BYOD upload'
else:
    cloze = 'The capital of France is [MASK].'
    expected_token = 'paris'
    sentences = [
        'The cat sat on the mat.',
        'A dog lay on the rug.',
        'Interest rates were raised by a quarter of a point.',
    ]
    sample_name = 'synthetic_cloze_and_sentences'
    sample_kind = 'synthetic (authored in this cell; the model card smoke cloze sentence)'
sentence_ids = [f's{index + 1}' for index in range(len(sentences))]
sample_sha256 = hashlib.sha256('\n'.join([cloze, *sentences]).encode('utf-8')).hexdigest()
print({'sample': sample_name, 'sample_kind': sample_kind, 'cloze': cloze, 'expected_token': expected_token, 'sentences': len(sentences), 'top_k': TOP_K, 'pooling': POOLING, 'text_sha256': sample_sha256})
for sentence_id, sentence in zip(sentence_ids, sentences, strict=True):
    print(f'{sentence_id}: {sentence[:100]}')

## 3. Validate the inputs against the pipeline ceilings

The pipeline enforces its ceilings inside `fill_mask` and `embed`; this cell imports the same constants from the package so the values shown are the ones in force, and checks the inputs before any model work, naming the failing condition and the corrective action. `MAX_TEXT_CHARS` is the character guard applied before tokenisation; `MAX_TEXT_TOKENS` (512, the checkpoint's position limit) is applied after tokenisation and **rejects** longer texts rather than truncating them, so it can only be confirmed by the model call; `MAX_BATCH` bounds one `embed` call; `MAX_TOP_K` bounds `top_k`; `MASK_TOKEN` must occur exactly once in the cloze sentence; `POOLINGS` names the two pooling policies; `VOCAB_SIZE` and `HIDDEN_SIZE` are the output-vocabulary and vector widths the contracts promise. The notebook does not trim or alter the texts.


In [ ]:
from bert_masked_lm_pipeline import DEFAULT_TOP_K, HIDDEN_SIZE, MASK_TOKEN, MAX_BATCH, MAX_TEXT_CHARS, MAX_TEXT_TOKENS, MAX_TOP_K, POOLINGS, VOCAB_SIZE

ceilings = {'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_TEXT_TOKENS': MAX_TEXT_TOKENS, 'MAX_BATCH': MAX_BATCH, 'MAX_TOP_K': MAX_TOP_K, 'VOCAB_SIZE': VOCAB_SIZE, 'HIDDEN_SIZE': HIDDEN_SIZE, 'MASK_TOKEN': MASK_TOKEN, 'POOLINGS': POOLINGS, 'DEFAULT_TOP_K': DEFAULT_TOP_K}
print(ceilings)
problems = []
if cloze.count(MASK_TOKEN) != 1:
    problems.append(f'cloze sentence contains {cloze.count(MASK_TOKEN)} {MASK_TOKEN} tokens, exactly one is required: edit the first line')
if not 1 <= TOP_K <= MAX_TOP_K:
    problems.append(f'TOP_K={TOP_K} is outside 1..MAX_TOP_K={MAX_TOP_K}: set a value in range')
if POOLING not in POOLINGS:
    problems.append(f'POOLING={POOLING!r} is not one of {POOLINGS}: choose a listed pooling')
if not 1 <= len(sentences) <= MAX_BATCH:
    problems.append(f'{len(sentences)} sentences is outside 1..MAX_BATCH={MAX_BATCH}: supply fewer lines or split the batch')
for name, text in [('cloze', cloze), *zip(sentence_ids, sentences, strict=True)]:
    if not text.strip():
        problems.append(f'{name} is empty: remove blank lines from the input')
    if len(text) > MAX_TEXT_CHARS:
        problems.append(f'{name} has {len(text)} chars > MAX_TEXT_CHARS={MAX_TEXT_CHARS}: shorten the text')
if problems:
    raise ValueError('input rejected before model execution: ' + '; '.join(problems))
print({'cloze_chars': len(cloze), 'sentences': len(sentences), 'longest_chars': max(len(text) for text in sentences), 'within_ceilings': True, 'token_ceiling': f'MAX_TEXT_TOKENS={MAX_TEXT_TOKENS} is checked by the pipeline after tokenisation and rejects, never truncates'})

## 4. Stage, verify and resolve the pinned model

Model acquisition goes through the package, not the notebook. The public API pins the exact upstream model repository and immutable 40-hex revision (`MODEL_ID`/`MODEL_REVISION` are imported from the package, never typed here) and refuses remote model code (`trust_remote_code=False`). The Git repository commits the DIMER snapshot manifest (`weights/bert-base-uncased/dimer-base-manifest.json`: model id, revision, and the byte size and SHA-256 of each of the 8 snapshot files) and the small config, tokenizer and licence files, but git-ignores the 440 MB `model.safetensors`, so a fresh clone must stage the missing file first. `stage_missing_files(WEIGHTS_DIR, allow_download=True)` fetches only the manifest-listed files that are absent, from the Hub at the pinned revision, into the repository's weights directory, and returns the list it fetched (`['model.safetensors']` on a fresh clone, `[]` when everything is already staged); it refuses to stage if the committed manifest disagrees with the package's pinned identity. `verify_snapshot(WEIGHTS_DIR)` then re-hashes every listed file and raises on the first size or digest mismatch; its returned dict is summarised. Only afterwards does `from_pretrained(weights_dir=WEIGHTS_DIR)` load tokenizer and model from that verified directory with `local_files_only=True` — there is no fallback to a different download. The loader reports that the checkpoint's pooler and next-sentence weights are unused by the masked-LM architecture; that notice is expected. The effective model identity and the selected device are printed before inference.


In [ ]:
from bert_masked_lm_pipeline import MODEL_ID, MODEL_KEY, MODEL_REVISION, BERTMaskedLMPipeline, stage_missing_files, verify_snapshot

print({'model_id': MODEL_ID, 'revision': MODEL_REVISION})
WEIGHTS_DIR = ROOT / 'weights' / MODEL_KEY
# Only the manifest-listed files that are absent are fetched, at the immutable revision the
# package pins; verify_snapshot then checks every byte count and SHA-256 before anything is loaded.
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'fetched': fetched, 'from': MODEL_ID, 'revision': MODEL_REVISION})
snapshot = verify_snapshot(WEIGHTS_DIR)
print({'snapshot_path': snapshot['path'], 'model_id': snapshot['modelId'], 'revision': snapshot['revision'], 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes')})
pipe = BERTMaskedLMPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': pipe.device, 'dtype': 'float32', 'source': pipe.source})

## 5. Fill the mask and read the candidates correctly

**Input/output contract.** `fill_mask(text, top_k=...)` takes one string with exactly one `[MASK]` and returns `candidates` — a list of `top_k` entries ordered by descending `score`, each with the WordPiece `token`, its `token_id`, the `score`, and the `sequence` with the mask substituted — plus `top_k`, `n_tokens` (WordPiece count including `[CLS]`/`[SEP]`), the device and the model identity. **Score semantics:** `score` is the softmax over the 30,522-token vocabulary at the masked position — a ranking signal that sums to 1 over the whole vocabulary, not a calibrated probability that the candidate is correct; the **default decision rule is `argmax`** (the first candidate) and the pipeline applies no minimum score, so a nonsensical sentence still yields a ranked list. Any acceptance threshold is owned by the caller and must be set on their own labelled cloze data.

**Evaluation:** the repository ships **no metric helper and reports no performance measure**. Fill-mask quality needs a labelled cloze set (sentence, mask position, gold token) over which the caller computes rank-based hit rates (top-1, top-5) with their own code, over enough sentences to state a dispersion. The synthetic sample has one author-expected token, so **no metric is reported**; the check below (the expected token appears in the top-`k`) is a falsifiable plumbing check on one sentence, not an accuracy result. The model card's CPU smoke on this sentence ranked `paris` first with score 0.4168 (9 tokens); that is one observation, not an expected value — near-tied candidates can reorder between CPU and CUDA kernels. Inference is deterministic on a fixed device and dtype (`model.eval()`, no sampling, no seed needed).


In [ ]:
import time

started = time.perf_counter()
filled = pipe.fill_mask(cloze, top_k=TOP_K)
fill_elapsed = time.perf_counter() - started
scores = [candidate['score'] for candidate in filled['candidates']]
fill_checks = {
    'top_k_candidates_returned': len(filled['candidates']) == TOP_K,
    'scores_descending': all(a >= b for a, b in zip(scores, scores[1:], strict=False)),
    'scores_in_unit_interval': all(0.0 < s <= 1.0 for s in scores),
    'tokens_within_vocab': all(0 <= candidate['token_id'] < VOCAB_SIZE for candidate in filled['candidates']),
    'n_tokens_within_ceiling': 1 <= filled['n_tokens'] <= MAX_TEXT_TOKENS,
}
if not all(fill_checks.values()):
    raise RuntimeError(f'fill_mask output failed a sanity check: {fill_checks}')
print({key: value for key, value in filled.items() if key != 'candidates'})
print({'seconds': round(fill_elapsed, 3), 'checks': fill_checks, 'decision_rule': 'argmax = first candidate; no threshold shipped'})
print(f'cloze: {cloze}')
for rank, candidate in enumerate(filled['candidates'], start=1):
    print(f"{rank:>2}. {candidate['token']:<12} id {candidate['token_id']:>6}  score {candidate['score']:.4f}  {candidate['sequence']}")
metrics = {}
if expected_token is not None:
    fill_sanity = {'expected_token_in_top_k': expected_token in [candidate['token'] for candidate in filled['candidates']]}
    print({'sanity_check': fill_sanity, 'note': 'falsifiable plumbing check on one synthetic sentence; not a metric'})
print('no metric is reported: the sample has one author-expected token, not a labelled cloze set, and the repository ships no metric helper; compute top-k hit rates on your own labelled sentences')

## 6. Embed the sentences and read the vectors correctly

**Input/output contract.** `embed(texts, pooling=...)` takes a list of 1..`MAX_BATCH` strings and returns `embeddings` — one list per input text, **in input order**, each of length `dim` (768) — plus `pooling` (`cls`: the `[CLS]` position of the last layer; `mean`: the attention-masked mean of the last layer, so padding never enters the average), `normalized` (`True`: every vector has unit L2 norm), `n_tokens` per text, the device and the model identity. The unit of embedding is **one vector per text**; there is no per-token or per-chunk output, and a text over `MAX_TEXT_TOKENS` is rejected rather than chunked or truncated. Missing data has no meaning here: empty strings are rejected, not embedded. **Embeddings are representations, not predictions:** they carry no label and no confidence, and their only meaning is relative — cosine between two vectors from the same model and the same pooling policy.

**No intrinsic metric exists** for an embedding: the repository ships no metric helper and no labelled data, and quality can only be judged through a downstream task with labels — a similarity benchmark with human judgements (Spearman correlation), or a retrieval or clustering set with relevance labels (recall@k). Below, the pairwise cosine matrix is computed from the returned vectors as a **qualitative check** that the contract works: on the default sample the two pet sentences are expected to score higher with each other than with the unrelated one. Cosine values are similarities in `[-1, 1]` on this model's geometry; raw BERT was not trained with a sentence-similarity objective, so its cosine scale is compressed and uncalibrated (the model card's smoke gave 0.8719 for the two pet sentences with mean pooling — one observation, not an expected value), absolute values are not comparable across models or pooling policies, and any "same meaning" threshold is the caller's to set on labelled pairs. Look for `(N, 768)` unit-norm vectors, token counts within the ceiling, and no truncation (the pipeline cannot truncate).


In [ ]:
import numpy as np

started = time.perf_counter()
embedding_result = pipe.embed(sentences, pooling=POOLING)
embed_elapsed = time.perf_counter() - started
vectors = np.asarray(embedding_result['embeddings'], dtype=np.float32)
norms = np.linalg.norm(vectors, axis=1)
embed_checks = {
    'one_vector_per_text': vectors.shape == (len(sentences), HIDDEN_SIZE),
    'dim_matches_contract': embedding_result['dim'] == HIDDEN_SIZE,
    'pooling_as_requested': embedding_result['pooling'] == POOLING,
    'unit_norm': bool(embedding_result['normalized']) and bool(np.allclose(norms, 1.0, atol=1e-4)),
    'all_values_finite': bool(np.isfinite(vectors).all()),
    'n_tokens_within_ceiling': all(1 <= n <= MAX_TEXT_TOKENS for n in embedding_result['n_tokens']),
}
if not all(embed_checks.values()):
    raise RuntimeError(f'embed output failed a sanity check: {embed_checks}')
print({key: value for key, value in embedding_result.items() if key != 'embeddings'})
print({'seconds': round(embed_elapsed, 3), 'shape': vectors.shape, 'norms': [round(float(n), 4) for n in norms], 'unit': 'one vector per text', 'checks': embed_checks})
cosine = vectors @ vectors.T
for sentence_id, row in zip(sentence_ids, cosine, strict=True):
    print(sentence_id, {other_id: round(float(value), 4) for other_id, value in zip(sentence_ids, row, strict=True)})
if not USE_BYOD:
    embed_sanity = {'related_pair_outscores_unrelated': bool(cosine[0, 1] > max(cosine[0, 2], cosine[1, 2]))}
    print({'sanity_check': embed_sanity, 'note': 'qualitative check on three synthetic sentences; not a metric'})
print('no metric is reported: embeddings are representations; the cosine table above is a qualitative check, and a similarity or retrieval score needs labelled data')

## 7. Export identifiers alongside vectors, and provenance

Two files are written under `outputs/`. The vectors go to CSV (`outputs/bert_masked_lm_embeddings.csv`) with one row per sentence — `id`, `n_tokens`, `pooling`, then `e0000…e0767` — so every vector stays attached to its identifier for downstream use. One JSON record (`outputs/bert_masked_lm_result.json`) preserves the fill-mask result (the cloze, every candidate with token, id, score, rank and sequence, the decision rule, `n_tokens`), the embedding contract (`dim`, `pooling`, `normalized`, unit), the identified sentences with their token counts, the cosine table keyed by identifier, the sanity checks, the ceilings in force, the empty metric block, the sample identity and digest, the repository revision, the model identifier and immutable revision, the verified snapshot summary, and the runtime identity (Python, `torch`, `transformers`, `numpy`, device, dtype). No credentials are involved in any step, so none can reach the export.


In [ ]:
import csv
import json

os.makedirs('outputs', exist_ok=True)
with open('outputs/bert_masked_lm_embeddings.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['id', 'n_tokens', 'pooling'] + [f'e{i:04d}' for i in range(HIDDEN_SIZE)])
    for sentence_id, vector, n_tokens in zip(sentence_ids, embedding_result['embeddings'], embedding_result['n_tokens'], strict=True):
        writer.writerow([sentence_id, n_tokens, embedding_result['pooling']] + [f'{value:.7f}' for value in vector])
payload = {
    'fill_mask': {
        'cloze': cloze,
        'expected_token': expected_token,
        'candidates': [{'rank': rank, **candidate} for rank, candidate in enumerate(filled['candidates'], start=1)],
        'decision_rule': 'argmax (first candidate); score is a vocabulary softmax, not a calibrated probability; no threshold shipped',
        'top_k': filled['top_k'],
        'n_tokens': filled['n_tokens'],
        'seconds': round(fill_elapsed, 3),
        'sanity_checks': fill_checks,
    },
    'embed': {
        'contract': {'dim': embedding_result['dim'], 'pooling': embedding_result['pooling'], 'normalized': embedding_result['normalized'], 'unit': 'one vector per text'},
        'texts': dict(zip(sentence_ids, sentences, strict=True)),
        'n_tokens': dict(zip(sentence_ids, embedding_result['n_tokens'], strict=True)),
        'cosine_similarity': {sentence_id: {other_id: float(value) for other_id, value in zip(sentence_ids, row, strict=True)} for sentence_id, row in zip(sentence_ids, cosine, strict=True)},
        'vectors_file': 'outputs/bert_masked_lm_embeddings.csv',
        'seconds': round(embed_elapsed, 3),
        'sanity_checks': embed_checks,
    },
    'ceilings': ceilings,
    'metrics': metrics,
    'sample': {'name': sample_name, 'kind': sample_kind, 'text_sha256': sample_sha256},
    'repository_revision': REPO_SHA,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'snapshot': {'path': snapshot['path'], 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes')},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'numpy': numpy.__version__,
        'device': pipe.device,
        'dtype': 'float32',
    },
}
with open('outputs/bert_masked_lm_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))
print('outputs/bert_masked_lm_result.json')

## Interpretation and limits

The fill-mask candidates are a softmax ranking over the vocabulary at one masked position: the first entry is the argmax, the `score` is not a calibrated probability, and the pipeline applies no threshold — the caller owns any cut-off and must set it on labelled cloze data. The embeddings are unit vectors in this model's 768-dimensional space: representations that predict nothing, whose only meaning is cosine between vectors from the same model and pooling policy; raw BERT's cosine scale is compressed and uncalibrated, and a retrieval-grade sentence encoder is a different model. On the synthetic sample both outputs are plumbing evidence only; no metric is reported because none can be computed without labelled data, and a real evaluation needs a labelled cloze set (top-k hit rates) and a judged similarity or retrieval set (Spearman, recall@k) with the caller's own code over enough items to state a dispersion. Text is lower-cased and accent-stripped by the tokenizer; texts over 512 WordPiece tokens are rejected, not truncated; the checkpoint is uncased English only and carries the gender and occupation associations the upstream card documents; the pipeline exposes no fine-tuning, generation, multi-mask filling or next-sentence prediction. Inference is deterministic on a fixed device and dtype, but CPU and CUDA kernels can reorder near-tied candidates and perturb low-order embedding digits.

Successful execution proves that the recorded repository revision can bootstrap in a fresh runtime, stage and digest-verify the pinned model snapshot, validate the demonstrated inputs against the enforced ceilings, execute both public pipeline paths, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, cloze accuracy or similarity quality on any domain, a usable threshold, safety for high-consequence decisions, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 4: a staged file is incomplete or altered — delete it from `weights/bert-base-uncased/` and rerun Section 4. A `ValueError` naming `[MASK]`, `MAX_TOP_K`, `MAX_BATCH` or `MAX_TEXT_CHARS` in Section 3: fix the form parameter or the BYOD lines and rerun from Section 2. A `ValueError` naming `MAX_TEXT_TOKENS` in Section 5 or 6: a text tokenises past 512 WordPiece tokens — shorten or split it; the pipeline never truncates. A loader notice that `bert.pooler` or `cls.seq_relationship` weights are unused in Section 4 is expected for the masked-LM architecture.

**Next experiments.** Change `POOLING` to `cls` and compare the cosine table with the `mean` one on the same sentences; raise `TOP_K` to 20 and inspect how quickly the scores decay; upload a dozen cloze sentences you can label yourself via `USE_BYOD` (one at a time — the pipeline fills one mask per call) and compute top-1/top-5 hit rate; embed the same sentence with and without capital letters to confirm the uncased tokenizer makes them identical. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: `../README.md`
- Repository model card: `../MODEL_CARD.md`
- Weight provenance: `../docs/WEIGHTS.md`
- Upstream model: https://huggingface.co/google-bert/bert-base-uncased
- Upstream code: https://github.com/google-research/bert
- BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding (Devlin et al., 2018): https://arxiv.org/abs/1810.04805
